# Statistical Properties of Financial Data — Lecture Notebook
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW
**Based on:** Brooks, C. — *Introductory Econometrics for Finance*, Cambridge University Press

---
**Learning Objectives:**
- Compute descriptive statistics with `describe()`, `skew()`, `kurtosis()`
- Recognise fat tails and asymmetry in real return data
- Run a formal normality test (Jarque-Bera) and interpret its p-value
- Identify the three universal **stylized facts**: volatility clustering, low ACF of returns, high ACF of squared returns
- Compare correlation in normal periods vs. crises

> This notebook accompanies the lecture slides. Run each cell with **Shift+Enter**.


## Step 0 — Install & Import Libraries

In [ ]:
!pip install yfinance seaborn statsmodels --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.graphics.tsaplots as tsaplots
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.3,'font.size':11
})
YELLOW='#FDE70E'; ORANGE='#FCB310'; RED='#C70101'; GREY='#4B4B4B'
print('✓ Libraries loaded.')

---
# Part 1 — Descriptive Statistics

**Always start here.** Before any model, look at the data.

| Statistic | Meaning | Finance Interpretation |
|-----------|---------|------------------------|
| **Mean** μ | Central tendency | Expected return |
| **Std dev** σ | Dispersion | Volatility / risk |
| **Min / Max** | Extremes | Worst / best day |
| **Quartiles** | 25%, 50%, 75% | Distribution shape |

### 1.1 Download Data and Run `describe()`

In [ ]:
TICKER = '^GSPC'   # S&P 500
START  = '2019-01-01'
END    = '2024-12-31'

data   = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)
prices = data['Close'].squeeze()
ret    = prices.pct_change().dropna() * 100   # in %

print(f'{TICKER}: {len(prices)} trading days, {prices.index[0].date()} → {prices.index[-1].date()}')
print('\nDescriptive Statistics (daily returns, %):')
print(ret.describe().round(4))

### 1.2 Annualisation — Why √252?

In [ ]:
TRADING_DAYS = 252

daily_mean = ret.mean()
daily_vol  = ret.std()

ann_return = daily_mean * TRADING_DAYS
ann_vol    = daily_vol  * np.sqrt(TRADING_DAYS)

print(f'Daily mean:   {daily_mean:.4f}%')
print(f'Daily vol:    {daily_vol:.4f}%')
print(f'Annual mean:  {ann_return:.2f}%   (× 252)')
print(f'Annual vol:   {ann_vol:.2f}%   (× √252 ≈ × 15.87)')
print('\nWhy √252?  Variance scales linearly with time, so std dev scales with √time.')
print('If daily returns are i.i.d. with variance σ², then 252-day variance = 252·σ², and 252-day std = σ·√252.')

### 1.3 Compare Multiple Assets

In [ ]:
tickers = ['AAPL', 'MSFT', 'JPM', '^GSPC']
raw          = yf.download(tickers, start=START, end=END, auto_adjust=True, progress=False)
prices_panel = raw['Close'].copy()
prices_panel.columns = ['AAPL','MSFT','JPM','SP500']
prices_panel = prices_panel.dropna()
ret_panel    = prices_panel.pct_change().dropna() * 100   # in %

summary = pd.DataFrame({
    'Mean (daily %)': ret_panel.mean().round(4),
    'Std (daily %)':  ret_panel.std().round(4),
    'Min (%)':        ret_panel.min().round(2),
    'Max (%)':        ret_panel.max().round(2),
    'Annual Vol (%)': (ret_panel.std()*np.sqrt(252)).round(2)
})
print('Daily Return Statistics 2019–2024:')
print(summary)

---
# Part 2 — Beyond Mean & Variance: Skewness, Kurtosis, Jarque-Bera

Mean and std describe a Normal distribution completely. Real returns are **not Normal** — we need two more moments.

| Moment | Measures | Normal value | Finance pattern |
|--------|----------|--------------|------------------|
| 3rd — **Skewness** | Asymmetry | 0 | < 0: more large losses than gains |
| 4th — **Excess Kurtosis** | Tail heaviness | 0 | > 0: extremes too frequent |

### 2.1 Skewness and Kurtosis for Real Returns

In [ ]:
moments = pd.DataFrame({
    'Mean':     ret_panel.mean().round(4),
    'Std':      ret_panel.std().round(4),
    'Skewness': ret_panel.skew().round(3),
    'Kurtosis': ret_panel.kurtosis().round(3),   # pandas returns *excess* kurtosis
})
print('Higher Moments — Daily Returns 2019–2024:')
print(moments)
print('\n→ Negative skewness:  large losses dominate large gains (crash risk)')
print('→ Excess kurtosis ≫ 0:  fat tails — extreme moves more frequent than Normal predicts')

### 2.2 Histogram vs. Normal Overlay — Visualising Fat Tails

In [ ]:
sp = ret_panel['SP500']
mu, sd = sp.mean(), sp.std()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# A: linear scale — see the peak
axes[0].hist(sp, bins=80, density=True, color=GREY, alpha=0.7, label='S&P 500 returns')
x = np.linspace(sp.min(), sp.max(), 500)
axes[0].plot(x, stats.norm.pdf(x, mu, sd), color=RED, lw=2, label='Normal(μ,σ)')
axes[0].set_title('A: Linear scale — peak is too tall', fontweight='bold')
axes[0].set_xlabel('Daily return (%)'); axes[0].set_ylabel('Density'); axes[0].legend()

# B: log scale — see the tails
axes[1].hist(sp, bins=80, density=True, color=GREY, alpha=0.7, label='S&P 500 returns')
axes[1].plot(x, stats.norm.pdf(x, mu, sd), color=RED, lw=2, label='Normal(μ,σ)')
axes[1].set_yscale('log')
axes[1].set_title('B: Log scale — tails are FAT', fontweight='bold')
axes[1].set_xlabel('Daily return (%)'); axes[1].set_ylabel('Density (log)'); axes[1].legend()

plt.suptitle('Empirical vs. Normal — Two Views', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print(f'Worst day:  {sp.min():.2f}%  →  {(sp.min()-mu)/sd:.2f} standard deviations from mean')
print(f'Under Normal, a move this extreme should occur roughly once every {1/(2*stats.norm.cdf((sp.min()-mu)/sd)):.0f} days')
print(f'Sample size: only {len(sp)} days — yet it happened.')

### 2.3 Q-Q Plot — The Diagnostic of Choice

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

stats.probplot(sp, dist='norm', plot=axes[0])
axes[0].set_title('S&P 500 — Q-Q Plot vs. Normal', fontweight='bold')
axes[0].get_lines()[0].set_markerfacecolor(GREY)
axes[0].get_lines()[0].set_markeredgecolor(GREY)
axes[0].get_lines()[0].set_markersize(3)
axes[0].get_lines()[1].set_color(RED)
axes[0].get_lines()[1].set_linewidth(2)

# Reference: simulated Normal
np.random.seed(42)
sim = np.random.normal(mu, sd, len(sp))
stats.probplot(sim, dist='norm', plot=axes[1])
axes[1].set_title('Simulated Normal — for comparison', fontweight='bold')
axes[1].get_lines()[0].set_markerfacecolor(GREY)
axes[1].get_lines()[0].set_markeredgecolor(GREY)
axes[1].get_lines()[0].set_markersize(3)
axes[1].get_lines()[1].set_color(RED)
axes[1].get_lines()[1].set_linewidth(2)

plt.tight_layout(); plt.show()

print('How to read a Q-Q plot:')
print('  • Points on the red line  →  Normal')
print('  • Points curve below at the left  →  fat LEFT tail (large losses)')
print('  • Points curve above at the right →  fat RIGHT tail (large gains)')
print('  • Both ends curve away  →  fat tails on both sides (typical for finance)')

### 2.4 Jarque-Bera Test

A formal test combining skewness and excess kurtosis:
$$
JB = \frac{n}{6}\left(S^2 + \frac{K^2}{4}\right)
$$
Under $H_0$ (Normal), $JB \sim \chi^2_2$. Reject when $p < 0.05$.

In [ ]:
print('Jarque-Bera Test for Normality (H₀: returns are Normal)')
print('=' * 70)
print(f'{"Asset":<8} | {"Skew":>7} | {"ExKurt":>7} | {"JB stat":>10} | {"p-value":>10} | {"Decision":<15}')
print('-' * 70)
for col in ret_panel.columns:
    r = ret_panel[col]
    jb_stat, jb_p = stats.jarque_bera(r)
    decision = 'Reject H₀' if jb_p < 0.05 else 'Fail to reject'
    print(f'{col:<8} | {r.skew():>7.3f} | {r.kurtosis():>7.3f} | {jb_stat:>10.1f} | {jb_p:>10.2e} | {decision:<15}')

print('\n→ ALL major financial series reject normality.')
print('→ Why it matters: VaR models assuming Normal underestimate tail risk.')
print('  In 2008 several banks lost more in days than their 99.9% VaR predicted in years.')

---
# Part 3 — Stylized Fact #1: Volatility Clustering

> *Large changes tend to be followed by large changes — of either sign — and small changes by small changes.*  
> — Mandelbrot (1963)

Calm periods cluster. Turbulent periods cluster. This is **not random**.

### 3.1 Visualise Clustering

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                          gridspec_kw={'height_ratios':[2,1], 'hspace':0.05})
axes[0].plot(prices.index, prices.values, color='black', lw=1.2)
axes[0].fill_between(prices.index, prices.values, alpha=0.06, color='black')
axes[0].set_ylabel('Price'); axes[0].set_title('S&P 500 — Price & Daily Returns', fontweight='bold')

# colour bars by sign — clustering becomes visually obvious
axes[1].bar(ret.index, ret.values,
            color=[RED if v < 0 else 'black' for v in ret.values], width=1.0)
axes[1].axhline(0, color=GREY, lw=0.8)
axes[1].set_ylabel('Daily return (%)'); axes[1].set_xlabel('Date')

# annotate COVID
covid_date = pd.Timestamp('2020-03-16')
axes[1].axvspan(pd.Timestamp('2020-02-20'), pd.Timestamp('2020-04-30'),
                color=ORANGE, alpha=0.15, label='COVID crash')
axes[1].legend(loc='lower left')

plt.tight_layout(); plt.show()
print('→ Big bars cluster around early 2020, late 2022, etc.  Quiet periods also cluster.')
print('→ Implication: today\'s volatility is informative about tomorrow\'s.')

---
# Part 4 — Stylized Facts #2 & #3: ACF and Rolling Volatility

**Stylized Fact #2:** Returns themselves have **almost no autocorrelation** (markets are roughly efficient).

**Stylized Fact #3:** **Squared** returns have **strong, persistent** autocorrelation — risk *is* predictable.

This combination is the empirical motivation for **GARCH** models (Lecture 11).

### 4.1 ACF of Returns vs. Squared Returns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

tsaplots.plot_acf(ret, lags=30, ax=axes[0], color='black', vlines_kwargs={'colors':'black'})
axes[0].set_title('ACF of Returns — almost zero', fontweight='bold')
axes[0].set_ylim(-0.15, 0.4)

tsaplots.plot_acf(ret**2, lags=30, ax=axes[1], color=RED, vlines_kwargs={'colors':RED})
axes[1].set_title('ACF of Squared Returns — high & persistent', fontweight='bold')
axes[1].set_ylim(-0.15, 0.4)

plt.tight_layout(); plt.show()

print('Reading the plots:')
print('  Left:  spikes inside the blue band → not significantly different from zero')
print('         → tomorrow\'s return direction is essentially unpredictable')
print('  Right: spikes well above the band, decaying slowly')
print('         → tomorrow\'s SQUARED return (≈ variance) IS predictable from today\'s')
print('         → this is exactly what GARCH models capture')

### 4.2 Rolling Volatility — How Risk Changes Over Time

In [ ]:
WINDOW = 21   # ≈ 1 trading month
roll_vol = ret.rolling(WINDOW).std() * np.sqrt(252)   # annualised

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(roll_vol.index, roll_vol.values, color=RED, lw=1.4)
ax.fill_between(roll_vol.index, 0, roll_vol.values, color=RED, alpha=0.15)
ax.axhline(roll_vol.mean(), color=GREY, ls='--', lw=1, label=f'Average: {roll_vol.mean():.1f}%')
ax.set_ylabel('Annualised volatility (%)')
ax.set_title(f'S&P 500 — Rolling {WINDOW}-Day Volatility (Annualised)', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

print(f'Lowest vol period: {roll_vol.idxmin().date()}  ({roll_vol.min():.1f}%)')
print(f'Highest vol period: {roll_vol.idxmax().date()}  ({roll_vol.max():.1f}%)')
print('\n→ Volatility is FAR from constant.  Models that assume constant σ (e.g. textbook Black-Scholes) miss this.')

---
# Part 5 — Correlation: Normal vs. Crisis

Diversification works in normal times. But correlations **rise sharply during crises** — often exactly when you need diversification most.  
This is called **correlation breakdown**.

### 5.1 Full-Sample Correlation Matrix

In [ ]:
corr_full = ret_panel.corr()
print('Correlation matrix 2019–2024:')
print(corr_full.round(3))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_full, annot=True, fmt='.2f', cmap='RdYlGn_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label':'Correlation'})
ax.set_title('Daily Return Correlations — Full Sample', fontweight='bold')
plt.tight_layout(); plt.show()

### 5.2 Crisis (Feb–May 2020) vs. Normal (2021)

In [ ]:
crisis = ret_panel.loc['2020-02-01':'2020-05-31']
normal = ret_panel.loc['2021-01-01':'2021-12-31']

corr_crisis = crisis.corr()
corr_normal = normal.corr()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, mat, title in [(axes[0], corr_normal, f'Normal: 2021 ({len(normal)} days)'),
                        (axes[1], corr_crisis, f'Crisis: Feb–May 2020 ({len(crisis)} days)')]:
    sns.heatmap(mat, annot=True, fmt='.2f', cmap='RdYlGn_r', center=0,
                vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
                cbar_kws={'label':'Correlation'})
    ax.set_title(title, fontweight='bold')
plt.tight_layout(); plt.show()

def avg_offdiag(c):
    m = c.values
    n = m.shape[0]
    return (m.sum() - n) / (n*(n-1))

print(f'Average off-diagonal correlation:')
print(f'  Normal (2021):       {avg_offdiag(corr_normal):.3f}')
print(f'  Crisis (Feb–May 20): {avg_offdiag(corr_crisis):.3f}')
print('\n→ Diversification benefits SHRINK exactly when you need them most.')
print('→ This is why portfolio risk models must account for time-varying correlations.')

---
## Summary Table

| Concept | Key Formula | Python |
|---------|-------------|--------|
| Descriptive stats | mean, std, quartiles | `ret.describe()` |
| Skewness | $E[(R-\mu)^3]/\sigma^3$ | `ret.skew()` |
| Excess kurtosis | $E[(R-\mu)^4]/\sigma^4 - 3$ | `ret.kurtosis()` |
| Annualisation | $\sigma_{ann} = \sigma_{daily}\sqrt{252}$ | `ret.std()*np.sqrt(252)` |
| Jarque-Bera | $JB = (n/6)(S^2 + K^2/4)$ | `stats.jarque_bera(ret)` |
| Volatility clustering | visual + ACF of $r^2$ | `tsaplots.plot_acf(ret**2)` |
| Rolling vol | windowed std × √252 | `ret.rolling(21).std()*np.sqrt(252)` |
| Crisis correlation | conditional corr | `ret_panel.loc[a:b].corr()` |

**Three Universal Stylized Facts:**
1. **Volatility clusters** — turbulence comes in bunches
2. **Returns have ~zero ACF** — direction is unpredictable
3. **Squared returns have high ACF** — risk *is* predictable → GARCH

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*Next: Loading & Visualising Financial Data in Google Colab — Live Demo*